In [ ]:
import helpy
import pandas as pd
from single_analysis import Analysis
import warnings
from sklearn.exceptions import ConvergenceWarning
from pandas.errors import PerformanceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=PerformanceWarning)

pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 10)

In [2]:
analysis_df = helpy.get_analysis_df(.08, 100, 3)

In [3]:
# list of models to test effectiveness of
training_range = (2007, 2020)
testing_range = (2021, 2025)

base_analysis = Analysis(analysis_df, training_range=training_range, testing_range=testing_range, config_file='all_predictor_tuning.json')


base_analysis.get_predictors()

In [4]:
base_analysis.predictors

{'Residuals': <fip_predictor.FIP_Predictor at 0x7928643673e0>,
 'Usage': <fip_predictor.FIP_Predictor at 0x7928bde79700>,
 'Regular Movement': <fip_predictor.FIP_Predictor at 0x79286bb0fc50>,
 'Baseline': <fip_predictor.FIP_Predictor at 0x7928643def60>,
 'No FIP': <fip_predictor.FIP_Predictor at 0x7928643ddd30>,
 'Velo + FIP': <fip_predictor.FIP_Predictor at 0x7928643deb40>}

In [5]:
base_analysis.predictor_results.sort_values('R2', ascending=False)

,Feature Set,Model,MAE,RMSE,R2
279,Regular Movement,Elastic Net,12.111350,15.197268,0.279666
5,Residuals,Elastic Net,12.110159,15.198503,0.279549
278,Regular Movement,Elastic Net,12.113384,15.199402,0.279464
4,Residuals,Elastic Net,12.111387,15.199844,0.279422
3,Residuals,Elastic Net,12.113019,15.201367,0.279278
...,...,...,...,...,...
500,Baseline,Random Forest,14.456778,17.966836,-0.006806
520,Baseline,Random Forest,14.477709,17.996700,-0.010156
529,Baseline,Random Forest,14.560944,18.094941,-0.021215
509,Baseline,Random Forest,14.601193,18.110651,-0.022989


In [6]:
base_analysis.get_lppm_scores()

In [7]:
base_analysis.results_df

,PlayerName,Season,FIP-,ERA-,pivFA,...,Velo + FIP48_rf_score,pred_Velo + FIP49_rf,Velo + FIP49_rf_score,pred_Velo + FIP0_gam,Velo + FIP0_gam_score
18,Aaron Civale,2021,114.279639,90.588031,91.495320,...,4.875196,106.025255,4.237928,110.070299,0.192884
19,Aaron Civale,2023,84.753358,82.963375,91.925195,...,1.920915,91.780151,0.144757,93.723429,-1.798521
20,Aaron Civale,2024,118.198572,110.490439,92.110243,...,0.711521,111.129906,-3.424047,106.553209,1.152650
21,Aaron Civale,2025,110.313836,118.279832,92.038485,...,-2.488626,112.335855,-6.018885,105.747393,0.569577
48,Aaron Nola,2021,77.268658,109.811032,93.058178,...,-1.657932,90.996988,-1.267657,88.380667,1.348664
...,...,...,...,...,...,...,...,...,...,...,...
3632,Zack Wheeler,2021,59.142138,65.997313,97.335892,...,1.713882,76.723673,0.395606,73.858020,3.261259
3633,Zack Wheeler,2022,71.175719,69.673653,96.103651,...,4.428255,79.376570,2.280495,78.546404,3.110662
3634,Zack Wheeler,2023,70.699745,81.310410,95.962228,...,3.389567,77.541994,3.793497,78.018913,3.316579
3635,Zack Wheeler,2024,75.354912,60.249158,95.467899,...,-0.471493,80.689134,7.332124,84.393429,3.627829


In [8]:
base_analysis.get_model_metrics()

,Model,N,Breakouts,ROC-AUC,PR-AUC
0,No FIP11_en,458,133,0.730827,0.577551
1,No FIP5_en,458,133,0.730480,0.576525
2,No FIP4_en,458,133,0.730411,0.576367
3,No FIP9_en,458,133,0.730503,0.576221
4,No FIP3_en,458,133,0.730480,0.576170
...,...,...,...,...,...
817,Baseline0_gam,458,133,0.500000,0.290393
818,Baseline36_xgb,458,133,0.490388,0.288909
819,Baseline30_xgb,458,133,0.479792,0.274280
820,Baseline48_xgb,458,133,0.469057,0.274277


<Figure size 1000x600 with 0 Axes>

In [9]:
base_analysis.results_df[['PlayerName', 'Season', 'No FIP11_en_score', 'FIP-', 'led_to_breakout']].sort_values('No FIP11_en_score', ascending=False)

,PlayerName,Season,No FIP11_en_score,FIP-,led_to_breakout
1157,Gavin Williams,2025,19.472610,107.075247,True
2305,MacKenzie Gore,2023,19.226406,111.273603,True
3126,Shane Baz,2025,19.066808,103.954689,True
505,Carlos Rodón,2024,18.340804,104.992095,True
1269,Hunter Greene,2022,17.755135,102.804968,True
...,...,...,...,...,...
696,Clayton Kershaw,2022,-18.280509,63.841059,False
639,Chris Sale,2024,-18.366917,52.975101,False
2506,Michael King,2023,-19.221803,71.702132,False
3194,Spencer Strider,2022,-23.613879,47.624203,False


In [10]:
helpy.breakout_rate_suite(base_analysis.results_df, 'No FIP11_en_score', 'led_to_breakout')

Overall Breakout Rate: 29.04%
Top 5% Breakout Rate: 81.82%
Top 10% Breakout Rate: 66.67%
Top 25% Breakout Rate: 56.14%
Top 50% Breakout Rate: 42.36%


In [11]:
baseline_scores_df = helpy.create_baseline_breakout_model(analysis_df, training_range, testing_range)

In [12]:
baseline_scores_df[['score_fip', 'FIP-', 'led_to_breakout']].sort_values('score_fip', ascending=False)

,score_fip,FIP-,led_to_breakout
3165,0.443510,119.136311,False
139,0.443506,119.210677,True
215,0.443506,118.807209,False
2513,0.443500,119.299546,False
1325,0.443480,119.480978,False
...,...,...,...
2850,0.227167,62.124544,False
1139,0.226949,63.995447,False
2037,0.226942,63.121288,False
696,0.226934,63.841059,False


In [13]:
helpy.breakout_rate_suite(baseline_scores_df, 'score_fip', 'led_to_breakout')

Overall Breakout Rate: 29.04%
Top 5% Breakout Rate: 27.27%
Top 10% Breakout Rate: 42.22%
Top 25% Breakout Rate: 45.61%
Top 50% Breakout Rate: 38.86%


In [14]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [15]:
two_year_breakout_df = helpy.is_breakout(base_analysis.results_df, .08, 100, 3)
base_analysis.get_model_metrics(df=two_year_breakout_df)

,Model,N,Breakouts,ROC-AUC,PR-AUC
0,No FIP11_en,458,133,0.730827,0.577551
1,No FIP5_en,458,133,0.730480,0.576525
2,No FIP4_en,458,133,0.730411,0.576367
3,No FIP9_en,458,133,0.730503,0.576221
4,No FIP3_en,458,133,0.730480,0.576170
5,No FIP10_en,458,133,0.730711,0.575999
6,No FIP2_en,458,133,0.730411,0.575932
7,No FIP1_en,458,133,0.730295,0.575375
8,No FIP0_en,458,133,0.730156,0.575018
9,No FIP17_en,458,133,0.725807,0.572024


<Figure size 1000x600 with 0 Axes>

In [16]:
one_year_breakout_df = helpy.is_breakout(base_analysis.results_df, .08, 100, 3)
base_analysis.get_model_metrics(df=one_year_breakout_df)

,Model,N,Breakouts,ROC-AUC,PR-AUC
0,No FIP11_en,458,133,0.730827,0.577551
1,No FIP5_en,458,133,0.730480,0.576525
2,No FIP4_en,458,133,0.730411,0.576367
3,No FIP9_en,458,133,0.730503,0.576221
4,No FIP3_en,458,133,0.730480,0.576170
5,No FIP10_en,458,133,0.730711,0.575999
6,No FIP2_en,458,133,0.730411,0.575932
7,No FIP1_en,458,133,0.730295,0.575375
8,No FIP0_en,458,133,0.730156,0.575018
9,No FIP17_en,458,133,0.725807,0.572024


<Figure size 1000x600 with 0 Axes>

In [17]:
two_year_breakout_df = helpy.is_breakout(base_analysis.results_df, .10, 100, 2)
base_analysis.get_model_metrics(df=two_year_breakout_df)

,Model,N,Breakouts,ROC-AUC,PR-AUC
0,No FIP5_en,458,117,0.729754,0.516043
1,No FIP4_en,458,117,0.729704,0.515947
2,No FIP11_en,458,117,0.729980,0.515837
3,No FIP3_en,458,117,0.729729,0.515629
4,No FIP2_en,458,117,0.729629,0.514946
5,No FIP1_en,458,117,0.729579,0.514663
6,No FIP0_en,458,117,0.729478,0.514482
7,No FIP10_en,458,117,0.729879,0.513325
8,No FIP9_en,458,117,0.729754,0.511191
9,No FIP8_en,458,117,0.728501,0.505684


<Figure size 1000x600 with 0 Axes>

In [18]:
two_year_breakout_df = helpy.is_breakout(base_analysis.results_df, .20, 100, 2)
base_analysis.get_model_metrics(df=two_year_breakout_df)

,Model,N,Breakouts,ROC-AUC,PR-AUC
0,Residuals27_rf,458,47,0.749858,0.288716
1,Residuals25_rf,458,47,0.707149,0.282420
2,No FIP27_rf,458,47,0.732567,0.279764
3,No FIP30_rf,458,47,0.728322,0.279252
4,No FIP24_rf,458,47,0.732878,0.278143
5,Residuals41_rf,458,47,0.723456,0.275791
6,No FIP12_rf,458,47,0.720557,0.274948
7,Residuals30_rf,458,47,0.746907,0.274354
8,No FIP13_rf,458,47,0.725786,0.273311
9,No FIP5_rf,458,47,0.727753,0.272117


<Figure size 1000x600 with 0 Axes>

In [19]:
two_year_breakout_df = helpy.is_breakout(base_analysis.results_df, .10, 110, 2)
base_analysis.get_model_metrics(df=two_year_breakout_df)

,Model,N,Breakouts,ROC-AUC,PR-AUC
0,No FIP11_en,458,130,0.754315,0.577745
1,No FIP9_en,458,130,0.755324,0.576736
2,No FIP10_en,458,130,0.754855,0.576469
3,No FIP0_en,458,130,0.753518,0.575833
4,No FIP1_en,458,130,0.753447,0.575789
5,No FIP4_en,458,130,0.753189,0.575622
6,No FIP5_en,458,130,0.753096,0.575520
7,No FIP3_en,458,130,0.753283,0.575398
8,No FIP2_en,458,130,0.753283,0.575281
9,No FIP8_en,458,130,0.754972,0.573530


<Figure size 1000x600 with 0 Axes>

In [20]:
two_year_breakout_df = helpy.is_breakout(base_analysis.results_df, .15, 100, 2)
base_analysis.get_model_metrics(df=two_year_breakout_df)

,Model,N,Breakouts,ROC-AUC,PR-AUC
0,No FIP5_en,458,76,0.737738,0.405858
1,No FIP4_en,458,76,0.737634,0.405844
2,No FIP3_en,458,76,0.737565,0.405679
3,No FIP1_en,458,76,0.737359,0.403755
4,No FIP2_en,458,76,0.737359,0.403750
5,No FIP11_en,458,76,0.738392,0.403747
6,No FIP0_en,458,76,0.737255,0.402329
7,No FIP10_en,458,76,0.738289,0.402105
8,Residuals27_rf,458,76,0.739563,0.401196
9,No FIP9_en,458,76,0.737979,0.398058


<Figure size 1000x600 with 0 Axes>

In [21]:
noFIP_predictions = base_analysis.predictors['No FIP'].predictions
nofip5_en = [d for d in noFIP_predictions if d.get("model_name") == "No FIP5_en"]
nofip11_en = [d for d in noFIP_predictions if d.get("model_name") == "No FIP11_en"]
nofip27_rf = [d for d in noFIP_predictions if d.get("model_name") == "No FIP27_rf"]
nofip7_xgb = [d for d in noFIP_predictions if d.get("model_name") == "No FIP7_xgb"]


residuals_predictions = base_analysis.predictors['Residuals'].predictions
residuals24_rf = [d for d in residuals_predictions if d.get("model_name") == "Residuals24_rf"]
residuals21_xgb = [d for d in residuals_predictions if d.get("model_name") == "Residuals21_xgb"]

In [22]:
print(f'no FIP 5 Elastic Net hyperparameters: {nofip5_en[0]['hyperparameters']}')
print(f'no FIP 11 Elastic Net hyperparameters: {nofip11_en[0]['hyperparameters']}')
print(f'no FIP 27 Random Forest hyperparameters: {nofip27_rf[0]['hyperparameters']}')
print(f'Residuals Random Forest hyperparameters: {residuals24_rf[0]['hyperparameters']}')
print(f'no FIP 27 XGBoost hyperparameters: {nofip7_xgb[0]['hyperparameters']}')
print(f'Residuals XGBoost hyperparameters: {residuals21_xgb[0]['hyperparameters']}')


no FIP 5 Elastic Net hyperparameters: {'l1_ratio': 1.0, 'alpha': 0.001}
no FIP 11 Elastic Net hyperparameters: {'l1_ratio': 1.0, 'alpha': 0.01}
no FIP 27 Random Forest hyperparameters: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 5}
Residuals Random Forest hyperparameters: {'n_estimators': 200, 'min_samples_split': 20, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 5}
no FIP 27 XGBoost hyperparameters: {'subsample': 0.8, 'reg_lambda': 20, 'reg_alpha': 1, 'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.01, 'colsample_bytree': 0.6}
Residuals XGBoost hyperparameters: {'subsample': 1.0, 'reg_lambda': 10, 'reg_alpha': 1, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.03, 'colsample_bytree': 0.6}
